In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [3]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)[:10]


['subtype_for_TCGA-KIRC.tsv',
 'degs.txt',
 'cases_for_TCGA-LUSC.tsv',
 'subtype_for_TCGA-HNSC.tsv',
 'cases_for_TCGA-PCPG.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_0045349c-69d9-4306-a403-c9c1fa836644_sample_type_Primary_Tumor_stage_unknown_fileid_0e0df72c-33c0-4e4f-939c-a4d45a6e1ea3.tsv',
 'cases_for_TCGA-THYM.tsv',
 'cases_for_TCGA-PAAD.tsv',
 'cases_for_TCGA-LAML.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_011b9b2d-ebe5-42bf-9662-d922faccc7a1_sample_type_Primary_Tumor_stage_Stage_IA_fileid_89409cf3-e710-47fd-af3c-6f8f0dec7c90.tsv']

### Get all programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [7]:
force=False
verbose=True

program = 'TCGA'

df_psi = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

print(len(df_psi))
df_psi.head(12)


Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'
33


,pid,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma
3,TCGA-LGG,Brain,TCGA-LGG,Gliomas,Brain Lower Grade Glioma
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme
5,TCGA-BRCA,Breast,TCGA-BRCA,"Adnexal and Skin Appendage Neoplasms, Adenomas...",Breast Invasive Carcinoma
6,TCGA-LUAD,Bronchus and lung,TCGA-LUAD,"Ductal and Lobular Neoplasms, Cystic, Mucinous...",Lung Adenocarcinoma
7,TCGA-LUSC,Bronchus and lung,TCGA-LUSC,"Squamous Cell Neoplasms, Adenomas and Adenocar...",Lung Squamous Cell Carcinoma
8,TCGA-MESO,"Bronchus and lung, Heart, mediastinum, and pleura",TCGA-MESO,Mesothelial Neoplasms,Mesothelioma
9,TCGA-CESC,"Cervix uteri, Ovary",TCGA-CESC,"Squamous Cell Neoplasms, Cystic, Mucinous and ...",Cervical Squamous Cell Carcinoma and Endocervi...


In [8]:
df_psi.tail(3)

,pid,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

In [22]:
ipsi = 5
pid = df_psi.iloc[ipsi].pid
primary_site = df_psi.iloc[ipsi].primary_site

print(ipsi, pid, primary_site)

5 TCGA-BRCA Breast


- Brain tumors do NOT use AJCC staging (3 - TCGA-LGG Brain)

In [23]:
force=False
verbose=True

pid = df_psi.iloc[ipsi].pid
primary_site = df_psi.iloc[ipsi].primary_site

print(ipsi, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

print("df_cases", len(df_cases), "df_subt", len(df_subt))

5 TCGA-BRCA Breast
Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'
df_cases 1098 df_subt 21


### df_subt

In [24]:
print(len(df_subt))
df_subt

21


,pid,subtype_global,tumor_class,subtype_tissue,stage,n
0,TCGA-BRCA,other,other,other,Stage IIA,263
1,TCGA-BRCA,lobular,other,breast_lobular,unknown,223
2,TCGA-BRCA,other,other,other,Stage IIB,180
3,TCGA-BRCA,other,other,other,Stage IIIA,107
4,TCGA-BRCA,other,other,other,unknown,76
5,TCGA-BRCA,other,other,other,Stage IA,69
6,TCGA-BRCA,other,other,other,Stage I,67
7,TCGA-BRCA,other,other,other,Stage IIIC,29
8,TCGA-BRCA,other,other,other,Stage IIIB,18
9,TCGA-BRCA,adenocarcinoma_generic,adenocarcinoma,adenocarcinoma_generic,unknown,18


### df_cases

In [25]:
df_cases.head(3)

,primary_site,disease_type,case_id,diagnoses,pid,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac,sstage
0,Breast,Complex Epithelial Neoplasms,e3935ce4-64d3-4a66-ba11-d308b844b410,"[{'primary_diagnosis': 'Metaplastic carcinoma,...",TCGA-BRCA,other,unknown,"Metaplastic carcinoma, NOS",NaN,NaN,...,complex epithelial neoplasms,metaplastic carcinoma,other,other,other,ok,ambiguous,1,0.000911,missing
1,Breast,Ductal and Lobular Neoplasms,e3b555aa-7f0a-49c6-9b13-182c61a144c1,[{'primary_diagnosis': 'Infiltrating duct carc...,TCGA-BRCA,other,Stage IIIA,"Infiltrating duct carcinoma, NOS",NaN,NaN,...,ductal and lobular neoplasms,infiltrating duct carcinoma,other,other,other,ok,ambiguous,1,0.000911,III
2,Breast,Ductal and Lobular Neoplasms,17ca61a2-607a-45ff-88fa-ef72e80bf891,[{'primary_diagnosis': 'Infiltrating duct carc...,TCGA-BRCA,other,Stage IA,"Infiltrating duct carcinoma, NOS",NaN,NaN,...,ductal and lobular neoplasms,infiltrating duct carcinoma,other,other,other,ok,ambiguous,1,0.000911,I


In [26]:
cols = ["pid", "case_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"]

df_cases[cols].head(12)

,pid,case_id,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-BRCA,e3935ce4-64d3-4a66-ba11-d308b844b410,other,other,other,unknown
1,TCGA-BRCA,e3b555aa-7f0a-49c6-9b13-182c61a144c1,other,other,other,Stage IIIA
2,TCGA-BRCA,17ca61a2-607a-45ff-88fa-ef72e80bf891,other,other,other,Stage IA
3,TCGA-BRCA,7d681cc6-689d-41c8-9e84-e13733089ec9,other,other,other,Stage IIB
4,TCGA-BRCA,17f275c1-a0d4-487d-8f02-ea279584b4cd,other,other,other,Stage IA
5,TCGA-BRCA,189e1f27-7738-413a-a4d4-97d41d592a13,other,other,other,Stage IIA
6,TCGA-BRCA,95c53ecf-d8f1-4bcb-9b1a-c9a0542939f0,other,other,other,Stage IA
7,TCGA-BRCA,c0f88fc5-56b0-4255-bbd1-9a21ced8b37b,other,other,other,Stage IIIA
8,TCGA-BRCA,0f64edec-0f1f-4025-8a53-75f9534f7828,other,other,other,Stage IIA
9,TCGA-BRCA,c25898b4-eb33-42f4-bf3f-bf532a929e7d,lobular,other,breast_lobular,unknown


### Samples

In [30]:
force=True
force=False
verbose=True

isubt=0

row = df_subt.iloc[isubt]

pid = row.pid
subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue
stage = None # row.stage

# s_case = f"{pid} subtype '{subtype_global}' tumor '{tumor_class}' subtype_tissue '{subtype_tissue}' stage '{stage}'"
# print(s_case)
df_samples = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                             tumor_class=tumor_class, subtype_tissue=subtype_tissue,
                                             batch_size=200, force=force, verbose=verbose)

print(len(df_samples))



Table opened ((1098, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-BRCA.tsv'
>>> TCGA-BRCA subtype 'other' tumor 'other' subtype_tissue 'other'
>>> 843 cases [001cef41-ff86-4d3f-a140-a647ac4b10a1,0045349c-69d9-4306-a403-c9c1fa836644,00a2d166-78c9-4687-a195-3d6315c27574] .....
Searching: 0-5 
>>> 5 ['001cef41-ff86-4d3f-a140-a647ac4b10a1', '0045349c-69d9-4306-a403-c9c1fa836644', '00a2d166-78c9-4687-a195-3d6315c27574', '011b9b2d-ebe5-42bf-9662-d922faccc7a1', '01263518-5f7c-49dc-8d7e-84b0c03a6a63']
....

👉 Returned 436 / Total paginated 436
Table saved ((587, 14)) at '/home/flavio/uv/perturb_agent/data/tcga/samples_for_TCGA-BRCA_subtype_'other'_tumor_'other'_subtype_tissue_'other'.tsv'
5-10 
>>> 5 ['0130d616-885e-4a6c-9d03-2f17dd692a05', '01674b2c-5cf2-478f-84a1-f69c39f47bd4', '016caf42-4e19-4444-ab5d-6cf1e76c4afa', '01b9f68b-4de4-4374-b4f2-8a0685e18153', '01eef340-598c-4205-a990-cec190ac2ca5']
.

👉 Returned 436 / Total paginated 436
Table saved ((587, 14)) at '/home/flav

### Available sample types

In [33]:
df_samples.sample_type.unique()

array(['Primary Tumor', 'Blood Derived Normal', 'Solid Tissue Normal'],
      dtype=object)

In [32]:
df2 = df_samples[df_samples.sample_type == 'Primary Tumor']
print(len(df_samples), len(df2))
df2.head()

587 361


,case_id,submitter_id,sample_id,sample_type,barcode_id,file_id,file_name,data_type,data_format,pid,subtype_global,tumor_class,subtype_tissue,stage
0,01263518-5f7c-49dc-8d7e-84b0c03a6a63,TCGA-A8-A07W,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,TCGA-A8-A07W-01A,7ccc5a5b-d3f1-46da-8540-d8fbb7c6c06e,TCGA-A8-A07W-01A-01-TSA.eae41004-b3f5-4528-9e9...,Slide Image,SVS,TCGA-BRCA,other,other,other,Stage IV
1,01263518-5f7c-49dc-8d7e-84b0c03a6a63,TCGA-A8-A07W,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,TCGA-A8-A07W-01A,8f355183-b72c-407e-8f25-f5eec7f115cc,e06580e9-0b4a-48f7-9522-d54229aefb2d.rna_seq.t...,Aligned Reads,BAM,TCGA-BRCA,other,other,other,Stage IV
2,01263518-5f7c-49dc-8d7e-84b0c03a6a63,TCGA-A8-A07W,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,TCGA-A8-A07W-01A,e209288d-df03-4a61-a84f-eec243a2c43f,TCGA-BRCA.3e06faf7-1e08-4c15-9aee-8891de8cad5c...,Gene Level Copy Number,TSV,TCGA-BRCA,other,other,other,Stage IV
3,01263518-5f7c-49dc-8d7e-84b0c03a6a63,TCGA-A8-A07W,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,TCGA-A8-A07W-01A,aa89d193-05a1-4f7f-9291-b9cf646656de,e06580e9-0b4a-48f7-9522-d54229aefb2d.rna_seq.a...,Gene Expression Quantification,TSV,TCGA-BRCA,other,other,other,Stage IV
4,01263518-5f7c-49dc-8d7e-84b0c03a6a63,TCGA-A8-A07W,5d1a5545-6934-41e5-b41b-8e0ee88e6ec6,Primary Tumor,TCGA-A8-A07W-01A,dd634319-6e80-4486-97b9-5d5fa3723a70,TCGA-BRCA.3cce14e4-4d6b-428c-812f-3716af4f855c...,Transcript Fusion,BEDPE,TCGA-BRCA,other,other,other,Stage IV


### GDC barcodes ~ submitter

In [34]:
barcodes = list(np.unique(df2.barcode_id))
len(barcodes)

11

In [35]:
barcodes

['TCGA-A1-A0SB-01A',
 'TCGA-A1-A0SB-01Z',
 'TCGA-A7-A26E-01A',
 'TCGA-A7-A26E-01B',
 'TCGA-A7-A26E-01Z',
 'TCGA-A8-A07W-01A',
 'TCGA-A8-A07W-01Z',
 'TCGA-AN-A0AM-01A',
 'TCGA-AN-A0AM-01Z',
 'TCGA-E2-A1IU-01A',
 'TCGA-E2-A1IU-01Z']

In [36]:
pid

'TCGA-BRCA'

In [37]:
mat = pid.lower().split('-')
study_id = mat[1] + '_' + mat[0]
print(">>>", study_id)

>>> brca_tcga


In [39]:
df_mut = gdc.get_mutations_from_samples(sample_ids=barcodes, study_id=study_id)
len(df_mut)

>>> https://www.cbioportal.org/api/molecular-profiles/brca_tcga_mutations/mutations/fetch


0

In [21]:
df_mut

""


In [40]:
sample_ids = barcodes

In [41]:
study_id

'brca_tcga'

In [ ]:
timeout = 60,
session: Optional[requests.Session] = None
url_cbioportal = "https://www.cbioportal.org/api"

sample_ids = [str(x).strip() for x in sample_ids if str(x).strip()]
if not sample_ids:
    raise ValueError("sample_ids is empty.")

molecular_profile_id = f"{study_id}_mutations"

http = session or requests.Session()

payload = {
    "sampleIds": sample_ids
}

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
}

resp = http.post(url_cbioportal, json=payload, headers=headers, timeout=timeout)

# Helpful error message from cBioPortal
if not resp.ok:
    msg = ""
    try:
        msg = resp.json()
    except Exception:
        msg = resp.text

    raise RuntimeError(
        f"cBioPortal request failed: HTTP {resp.status_code} | "
        f"profile={molecular_profile_id} | details={msg}"
    )

data = resp.json()

df = pd.DataFrame(data)

# Optional: reorder a few common columns if present
cols = ["sampleId", "patientId", "studyId", "molecularProfileId","gene",
        "entrezGeneId", "proteinChange", "mutationType", "mutationStatus",
        "variantType", "chr", "startPosition", "endPosition",
        "referenceAllele", "tumorSeqAllele2",
]
# the selected cols + others not listed
cols = [c for c in cols if c in df.columns] + [c for c in df.columns if c not in cols]


NameError: name 'timeout' is not defined

In [ ]:
df[cols]